## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### The `setof` Operator
#### AMPL Formulation

In [3]:
%%ampl_eval
reset;                          # clear previous model/data

set PROD;                       # set of products
param cost {PROD} >= 0;         # unit cost per product
var Make {PROD} >= 0;          # production quantity
var Sell {PROD} >= 0;          # sales quantity

# Extract unique cost values across products
set COST_VALUES = setof {p in PROD} cost[p];

# Parameter for market availability/limit per product
param market {PROD} >= 0;

# Select products with positive market presence
set ACTIVE_PROD = setof {p in PROD: market[p] > 0} p;

In [4]:
%%ampl_eval
data;                           # Start data section

set PROD := A B C D;            # Products in the model

param cost := A 10 B 5 C 8 D 12;# Cost per unit of each product
param market := A 1 B 0 C 3 D 0;# Market availability (used to determine active products)

display COST_VALUES, ACTIVE_PROD;

set COST_VALUES := 10 5 8 12;

set ACTIVE_PROD := A C;



#### amplpy Formulation

In [5]:
# Sets
ampl.set['PROD'] = ['A', 'B', 'C', 'D']

# Parameters (inline format)
ampl.param['cost'].set_values({'A': 10, 'B': 5, 'C': 8, 'D': 12})
ampl.param['market'].set_values({'A': 1, 'B': 0, 'C': 3, 'D': 0})

# Equivalent of COST_VALUES = setof {p in PROD} cost[p]
cost_values = set(ampl.param['cost'].get_values().to_dict().values())

# Equivalent of ACTIVE_PROD = setof {p in PROD: market[p] > 0} p
market_dict = ampl.param['market'].get_values().to_dict()
active_prod = {p for p, v in market_dict.items() if v > 0}

print("COST_VALUES =", cost_values)
print("ACTIVE_PROD =", active_prod)

COST_VALUES = {8, 10, 12, 5}
ACTIVE_PROD = {'C', 'A'}
